# Customer Churn Prediction

Model code: data transformations, training, selection and evaluation.

Code developed with assistance from **CODEX**. The test set was reused; OHE precedes CV, and tuning/threshold selection share folds. Results remain exploratory. [Methodology and reproduction](../Project%20Documentation/README.md).

## 1. Imports and data

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data/raw").is_dir() else Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data/raw/Telco-Customer-Churn.csv"
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == (
    "16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91"
)
df = pd.read_csv(DATA_PATH)
assert df.shape == (7043, 21)


## 2. Split, target and cleaning

In [ ]:
# Exclude the identifier and encode the target.
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})
assert y.notna().all()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# All 11 blank TotalCharges values belong to customers with tenure=0.
blank_charges = df["TotalCharges"].astype(str).str.strip().eq("")
assert blank_charges.sum() == 11 and df.loc[blank_charges, "tenure"].eq(0).all()
for data in (X_train, X_test):
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce").fillna(0)


## 3. Encode categories and scale LR features

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "str", "category"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_encoded = encoder.fit_transform(X_train[categorical_cols])
test_encoded = encoder.transform(X_test[categorical_cols])
encoded_cols = encoder.get_feature_names_out(categorical_cols)

# Preserve the original feature order: numeric columns, then OHE columns.
X_train_rf = pd.concat([
    X_train[numeric_cols],
    pd.DataFrame(train_encoded, columns=encoded_cols, index=X_train.index),
], axis=1)
X_test_rf = pd.concat([
    X_test[numeric_cols],
    pd.DataFrame(test_encoded, columns=encoded_cols, index=X_test.index),
], axis=1)

# RF remains unscaled. Only these three continuous variables are scaled for LR.
scale_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
X_train_lr, X_test_lr = X_train_rf.copy(), X_test_rf.copy()
scaler = StandardScaler()
X_train_lr[scale_cols] = scaler.fit_transform(X_train_lr[scale_cols])
X_test_lr[scale_cols] = scaler.transform(X_test_lr[scale_cols])


## 4. Train baseline models

In [ ]:
def evaluate(model, features, target, threshold=None):
    probability = model.predict_proba(features)[:, 1]
    # Default predict preserves the estimator's tie handling at 0.50.
    prediction = (
        model.predict(features) if threshold is None
        else (probability >= threshold).astype(int)
    )
    return {
        "Accuracy": accuracy_score(target, prediction),
        "Precision": precision_score(target, prediction),
        "Recall": recall_score(target, prediction),
        "F1-Score": f1_score(target, prediction),
        "ROC-AUC": roc_auc_score(target, probability),
    }

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}
datasets = {
    "Logistic Regression": (X_train_lr, X_test_lr),
    "Random Forest": (X_train_rf, X_test_rf),
}
results = {}
for name, model in models.items():
    train, test = datasets[name]
    model.fit(train, y_train)
    results[name] = evaluate(model, test, y_test)


## 5. Fit the oversampling comparison

In [ ]:
for name, model in models.items():
    train, test = datasets[name]
    train_os, target_os = RandomOverSampler(random_state=42).fit_resample(train, y_train)
    sampled_model = clone(model).fit(train_os, target_os)
    results[f"{name} + OS"] = evaluate(sampled_model, test, y_test)


## 6. Select LR hyperparameters on training data

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipeline = Pipeline([
    ("preprocessor", ColumnTransformer(
        [("scaler", StandardScaler(), scale_cols)], remainder="passthrough"
    )),
    ("model", LogisticRegression(max_iter=2000)),
])
grid_search = GridSearchCV(
    pipeline,
    param_grid={
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__class_weight": [None, "balanced"],
    },
    scoring={
        "accuracy": "accuracy", "precision": "precision", "recall": "recall",
        "f1": "f1", "roc_auc": "roc_auc",
    },
    refit="f1", cv=cv, n_jobs=-1, return_train_score=True,
)
grid_search.fit(X_train_rf, y_train)
cv_results = pd.DataFrame(grid_search.cv_results_)


## 7. Select the threshold from out-of-fold training predictions

In [ ]:
probability_cv = cross_val_predict(
    clone(grid_search.best_estimator_), X_train_rf, y_train,
    cv=cv, method="predict_proba", n_jobs=-1,
)[:, 1]
threshold_results = []
for threshold in np.round(np.arange(0.10, 0.91, 0.01), 2):
    prediction_cv = (probability_cv >= threshold).astype(int)
    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, prediction_cv, zero_division=0),
        "Recall": recall_score(y_train, prediction_cv, zero_division=0),
        "F1-Score": f1_score(y_train, prediction_cv, zero_division=0),
    })
threshold_df = pd.DataFrame(threshold_results)
best_threshold = float(threshold_df.loc[threshold_df["F1-Score"].idxmax(), "Threshold"])


## 8. Refit, evaluate and save the computed results

In [ ]:
final_model = clone(grid_search.best_estimator_).fit(X_train_rf, y_train)
results["Logistic Regression optimizada"] = evaluate(final_model, X_test_rf, y_test, best_threshold)
all_results = pd.DataFrame.from_dict(results, orient="index").rename_axis("Modelo")

report_dir = PROJECT_ROOT / "reports"
report_dir.mkdir(exist_ok=True)
all_results.to_csv(report_dir / "metrics.csv", float_format="%.10f")
cv_results.to_csv(report_dir / "cv_results.csv", index=False)
threshold_df.to_csv(report_dir / "threshold_results.csv", index=False)
print("Selected LR:", grid_search.best_params_, "Threshold:", best_threshold)
display(all_results.round(4))
